# 구조기반청킹

## Step 1 — PDF 진단 (약 30분)

### 1. 라이브러리 설치
- 확인할 것: 파일 크기가 정상적으로 출력되는지.

In [ ]:
# 셀 1: 라이브러리 설치
!pip install pdfplumber -q

### 2. Google Drive 연결 및 파일 확인

In [ ]:
# 셀 2: Google Drive 마운트 및 파일 존재 확인
from google.colab import drive
import os

drive.mount('/content/drive')

PDF_PATH = "/content/drive/MyDrive/IPCC_data/KO_IPCC_AR6_SYR_FullVolume.pdf"

# 파일 존재 여부 + 크기 확인
if os.path.exists(PDF_PATH):
    size_mb = os.path.getsize(PDF_PATH) / (1024 * 1024)
    print(f"✓ 파일 확인됨")
    print(f"  경로: {PDF_PATH}")
    print(f"  크기: {size_mb:.1f} MB")
else:
    print("✗ 파일을 찾을 수 없습니다. 경로를 다시 확인하세요.")

### 3. 페이지 수 및 기본 메타데이터 확인
- 확인할 것: 페이지 수가 170쪽 내외인지, 제목·저자 등 메타데이터 정상 출력 여부.

In [ ]:
# 셀 3: 기본 메타데이터 확인
import pdfplumber

with pdfplumber.open(PDF_PATH) as pdf:
    total_pages = len(pdf.pages)
    metadata   = pdf.metadata

print(f"총 페이지 수 : {total_pages}")
print()
print("── 메타데이터 ──────────────────────────")
for k, v in metadata.items():
    print(f"  {k:20s}: {v}")

### 4. 텍스트 추출 가능 여부 확인 (1~3페이지 샘플)
- 확인할 것: 한글이 정상 출력되는지, 아니면 빈칸/깨진 문자가 나오는지.
→ 빈칸이면 OCR이 필요하므로 Day 1 범위를 조정해야 합니다.

In [ ]:
# 셀 4: 텍스트 추출 가능 여부 — 앞 3페이지 샘플 확인
with pdfplumber.open(PDF_PATH) as pdf:
    for i in range(min(3, len(pdf.pages))):
        text = pdf.pages[i].extract_text() or ""
        print(f"{'─'*50}")
        print(f"[PAGE {i+1}] 글자 수: {len(text)}")
        print(text[:300] if text else "  ⚠ 텍스트 없음 (스캔본 또는 이미지 페이지 가능성)")
        print()

### 5. 전체 페이지 추출 품질 통계
- 확인할 것: 빈 페이지 비율이 전체의 20% 이하면 정상입니다. 빈 페이지 목록은 Day 2 청킹 로직에서 건너뛰기 처리에 사용합니다.

In [ ]:
# 셀 5: 전체 페이지 추출 품질 통계
with pdfplumber.open(PDF_PATH) as pdf:
    total = len(pdf.pages)
    char_counts = []

    for page in pdf.pages:
        text = page.extract_text() or ""
        char_counts.append(len(text))

empty_pages   = [i+1 for i, c in enumerate(char_counts) if c < 50]
rich_pages    = [i+1 for i, c in enumerate(char_counts) if c >= 50]
avg_chars     = sum(char_counts) / total

print(f"총 페이지          : {total}")
print(f"텍스트 있는 페이지 : {len(rich_pages)} 페이지")
print(f"텍스트 빈 페이지   : {len(empty_pages)} 페이지  ← 그림/표 페이지 추정")
print(f"평균 글자 수/페이지: {avg_chars:.0f} 자")
print()
if empty_pages:
    print(f"빈 페이지 목록: {empty_pages}")
else:
    print("빈 페이지 없음 — 전 페이지 텍스트 추출 가능")

### 6. 폰트 상태 확인 (한글 깨짐 예측)
- 확인할 것: emb 컬럼이 yes인 폰트가 대부분이면 정상입니다. no가 많으면 텍스트 추출 품질이 낮을 수 있습니다.

In [ ]:
# 셀 6: 폰트 상태 확인 — 한글 깨짐 예측
import subprocess

result = subprocess.run(
    ["pdffonts", PDF_PATH],
    capture_output=True, text=True
)

if result.returncode == 0:
    lines = result.stdout.strip().split("\n")
    print(f"폰트 수: {len(lines) - 2}개")
    print()
    # 헤더 + 처음 15개 폰트만 출력
    for line in lines[:17]:
        print(line)
    if len(lines) > 17:
        print(f"  ... 외 {len(lines)-17}개 폰트")
else:
    # pdffonts 없을 경우 설치 후 재실행
    print("pdffonts 설치 중...")
    subprocess.run(["apt-get", "install", "-y", "-q", "poppler-utils"])
    result = subprocess.run(["pdffonts", PDF_PATH], capture_output=True, text=True)
    print(result.stdout[:1500])

### 7. 진단 결과 요약 출력

In [ ]:
# 셀 7: 진단 결과 요약 — 메모리 문서에 복붙할 수 있는 형태로 출력
summary = f"""
╔══════════════════════════════════════════════════╗
  Day 1 — Step 1 PDF 진단 결과 요약
╚══════════════════════════════════════════════════╝

파일명  : KO_IPCC_AR6_SYR_FullVolume.pdf
총 페이지: {total}
텍스트 있는 페이지: {len(rich_pages)} / {total}
텍스트 빈 페이지  : {len(empty_pages)} / {total}
평균 글자 수/페이지: {avg_chars:.0f} 자
빈 페이지 목록    : {empty_pages if empty_pages else '없음'}

판정:
  {'✓ 텍스트 추출 정상 → Step 2 진행 가능' if len(empty_pages) < total * 0.3 else '⚠ 빈 페이지 비율 높음 → OCR 필요 여부 검토'}
"""
print(summary)

## Step 2 — 전체 텍스트 추출 (약 1시간)

### 1. 전체 텍스트 추출 (페이지별 저장)

In [ ]:
# 셀 1: 전체 텍스트 추출
import pdfplumber
from pathlib import Path

PDF_PATH   = "/content/drive/MyDrive/IPCC_data/KO_IPCC_AR6_SYR_FullVolume.pdf"
OUTPUT_DIR = "/content/drive/MyDrive/IPCC_data/parsed"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# Step 1에서 확인한 빈 페이지 목록
EMPTY_PAGES = {2,6,8,12,18,52,53,56,57,106,107,132,133,134,150,151,158,159,187}

pages_data = []   # {"page_num", "text", "char_count"}

with pdfplumber.open(PDF_PATH) as pdf:
    total = len(pdf.pages)
    for i, page in enumerate(pdf.pages):
        page_num = i + 1

        if page_num in EMPTY_PAGES:
            pages_data.append({"page_num": page_num, "text": "", "char_count": 0})
            continue

        text = page.extract_text() or ""
        pages_data.append({"page_num": page_num, "text": text, "char_count": len(text)})

        if page_num % 30 == 0:
            print(f"  진행 중... {page_num}/{total} 페이지")

print(f"✓ 추출 완료 — 총 {total} 페이지")
print(f"  텍스트 있음: {sum(1 for p in pages_data if p['char_count'] > 0)} 페이지")
print(f"  건너뜀     : {len(EMPTY_PAGES)} 페이지")

### 2. 텍스트 품질 이상 페이지 탐지
- 확인할 것: 글자 수가 적은 페이지는 목차·색인일 가능성이 높고, 많은 페이지는 표가 텍스트로 섞인 경우입니다. 이상하다 싶은 페이지 번호를 메모해두세요.

In [ ]:
# 셀 2: 이상 페이지 탐지 — 글자 수 기준 상하위 이상치 확인
char_counts = [p["char_count"] for p in pages_data if p["char_count"] > 0]
avg   = sum(char_counts) / len(char_counts)
sorted_counts = sorted(char_counts)

# 하위 5% / 상위 5% 기준값
low_threshold  = sorted_counts[int(len(sorted_counts) * 0.05)]
high_threshold = sorted_counts[int(len(sorted_counts) * 0.95)]

suspicious_low  = [p for p in pages_data if 0 < p["char_count"] < low_threshold]
suspicious_high = [p for p in pages_data if p["char_count"] > high_threshold]

print(f"평균 글자 수/페이지 : {avg:.0f} 자")
print(f"하위 5% 기준        : {low_threshold} 자")
print(f"상위 5% 기준        : {high_threshold} 자")
print()

print(f"── 글자 수 적은 페이지 (하위 5%, {len(suspicious_low)}개) ──")
for p in suspicious_low:
    preview = p["text"][:60].replace("\n", " ")
    print(f"  p.{p['page_num']:3d} ({p['char_count']:4d}자) | {preview}")

print()
print(f"── 글자 수 많은 페이지 (상위 5%, {len(suspicious_high)}개) ──")
for p in suspicious_high:
    preview = p["text"][:60].replace("\n", " ")
    print(f"  p.{p['page_num']:3d} ({p['char_count']:4d}자) | {preview}")

### 3. 샘플 페이지 육안 확인 (본문 구간)
- 확인할 것: 줄 구분이 자연스러운지, 헤딩처럼 보이는 줄이 어떤 형태인지 눈으로 확인합니다. Step 3 정규식 설계의 직접적인 근거가 됩니다.

In [ ]:
# 셀 3: 본문 구간 샘플 3페이지 육안 확인
# 빈 페이지 제외하고 앞/중간/뒤 각 1페이지 출력
sample_pages = [p for p in pages_data if p["char_count"] > 500]
targets = [
    sample_pages[10],                        # 앞부분
    sample_pages[len(sample_pages) // 2],    # 중간
    sample_pages[-10],                       # 뒷부분
]

for p in targets:
    print(f"{'═'*60}")
    print(f"[PAGE {p['page_num']}]  ({p['char_count']}자)")
    print(f"{'─'*60}")
    print(p["text"][:600])
    print()

### 4. 전체 텍스트 파일 저장
- 페이지 구분자 <<<PAGE_N>>> 형식으로 저장합니다. Day 2 청킹 로직에서 페이지 번호를 메타데이터로 추적할 때 이 구분자를 활용합니다.

In [ ]:
# 셀 4: 페이지 구분자 포함 전체 텍스트 저장
output_path = f"{OUTPUT_DIR}/ipcc_raw.txt"

lines = []
for p in pages_data:
    lines.append(f"\n<<<PAGE_{p['page_num']}>>>\n")
    if p["text"]:
        lines.append(p["text"])
    else:
        lines.append("[빈 페이지 — 그림/표 추정]")

Path(output_path).write_text("\n".join(lines), encoding="utf-8")

file_size_kb = Path(output_path).stat().st_size / 1024
print(f"✓ 저장 완료: {output_path}")
print(f"  파일 크기: {file_size_kb:.0f} KB")
print(f"  총 글자 수: {sum(p['char_count'] for p in pages_data):,} 자")

### 5. pages_data 직렬화 저장 (Day 2 재사용용)

In [ ]:
# 셀 5: pages_data를 JSON으로 저장 — Day 2에서 다시 추출 없이 바로 로드
import json

json_path = f"{OUTPUT_DIR}/ipcc_pages.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(pages_data, f, ensure_ascii=False, indent=2)

print(f"✓ 저장 완료: {json_path}")
print()
print("── Day 2 로드 방법 ──────────────────────────────")
print("import json")
print(f"with open('{json_path}') as f:")
print("    pages_data = json.load(f)")

### 6.  Step 2 결과 요약 출력

In [ ]:
# 셀 6: Step 2 결과 요약 — 메모리 문서 업데이트용
total_chars = sum(p["char_count"] for p in pages_data)
content_pages = [p for p in pages_data if p["char_count"] > 0]

print(f"""
╔══════════════════════════════════════════════════╗
  Day 1 — Step 2 텍스트 추출 결과 요약
╚══════════════════════════════════════════════════╝

총 페이지         : {len(pages_data)}
텍스트 추출 성공  : {len(content_pages)} 페이지
건너뛴 빈 페이지  : {len(EMPTY_PAGES)} 페이지
총 글자 수        : {total_chars:,} 자
평균 글자 수/페이지: {total_chars // len(content_pages):,} 자

저장 파일:
  ✓ ipcc_raw.txt      (페이지 구분자 포함 전문)
  ✓ ipcc_pages.json   (Day 2 청킹 재사용용)

다음 단계: Step 3 — 구조 패턴 분석 (정규식 설계)
""")

## Step 3 — 구조 패턴 분석 (약 1~2시간, Day 1의 핵심)

### 1. 전체 텍스트를 줄 단위 리스트로 변환

In [ ]:
# 셀 1: pages_data에서 줄 단위 리스트 생성
# (이미 pages_data가 메모리에 있으면 JSON 로드 불필요)
import json, re
from pathlib import Path

OUTPUT_DIR = "/content/drive/MyDrive/IPCC_data/parsed"

# pages_data가 없을 경우에만 로드
if 'pages_data' not in dir():
    with open(f"{OUTPUT_DIR}/ipcc_pages.json", encoding="utf-8") as f:
        pages_data = json.load(f)

# 줄 단위 리스트 생성 — (page_num, line_text) 튜플
lines_with_page = []
for p in pages_data:
    if not p["text"]:
        continue
    for line in p["text"].split("\n"):
        line = line.strip()
        if line:  # 빈 줄 제외
            lines_with_page.append((p["page_num"], line))

print(f"총 줄 수: {len(lines_with_page):,} 줄")
print()
print("── 샘플 20줄 (p.20 근처) ──────────────────────")
samples = [(pn, ln) for pn, ln in lines_with_page if pn == 20][:20]
for pn, ln in samples:
    print(f"  [p.{pn}] {ln}")

### 2. IPCC 문서 특화 헤딩 패턴 후보 탐색
- 확인할 것: 매칭 수가 0인 패턴은 이 문서에 없는 구조입니다. 그림/표 패턴 매칭 수가 많으면 오탐 필터가 중요해집니다.

In [ ]:
# 셀 2: IPCC 문서 특화 헤딩 패턴 후보 정의 및 탐색

HEADING_PATTERNS = {
    # ── IPCC 고유 구조 ──────────────────────────────
    "SPM섹션":      r"^SPM[.\s][A-Z]",              # SPM.A / SPM B
    "섹션번호":     r"^섹션\s*\d+",                  # 섹션 1, 섹션2
    "핵심발견":     r"^(핵심\s*(발견|메시지|질문))",  # 핵심발견, 핵심 메시지
    "Box":          r"^Box\s*[\d.]+",                # Box 1.1
    "CrossSection": r"^Cross[-\s]Section",           # Cross-Section
    "부속서":       r"^부속서\s*[IVX\d]+",           # 부속서 I, 부속서 V

    # ── 일반 계층 구조 ──────────────────────────────
    "숫자점숫자점숫자": r"^\d+\.\d+\.\d+\s+\S",      # 1.1.1 제목
    "숫자점숫자":    r"^\d+\.\d+\s+[가-힣A-Z]",      # 1.1 제목
    "숫자점":       r"^\d+\.\s+[가-힣A-Z]",          # 1. 제목
    "알파벳점":     r"^[A-Z]\.\s+[가-힣]",           # A. 제목

    # ── 특수 섹션 ───────────────────────────────────
    "자주묻는질문": r"^자주\s*묻는\s*질문",
    "용어해설":     r"^용어\s*(해설|설명|정의)",
    "참고문헌":     r"^참고\s*문헌",
    "그림":         r"^그림\s*[\d.]+",               # 그림 1.1 — 헤딩 아님, 오탐용
    "표":           r"^표\s*[\d.]+",                 # 표 1.1  — 헤딩 아님, 오탐용
}

# 패턴별 탐색
results = {name: [] for name in HEADING_PATTERNS}

for page_num, line in lines_with_page:
    for name, pat in HEADING_PATTERNS.items():
        if re.match(pat, line):
            results[name].append((page_num, line))
            break  # 중복 매칭 방지

# 결과 출력
print(f"{'패턴명':18s} {'매칭 수':>6s}  {'샘플 (최대 3개)'}")
print("─" * 80)
for name, hits in results.items():
    sample_str = " | ".join(f"p.{pn} 「{ln[:30]}」" for pn, ln in hits[:3])
    print(f"  {name:16s} {len(hits):5d}개  {sample_str}")

### 3. 각 패턴 실제 매칭 샘플 상세 확인
- 확인할 것: 각 패턴이 실제 헤딩을 잡고 있는지, 아니면 본문 내 숫자나 그림 캡션을 잡는지 판단합니다.

In [ ]:
# 셀 3: 패턴별 실제 매칭 내용 상세 출력 — 오탐 여부 육안 확인

# 확인할 패턴만 선택 (매칭 수 > 0인 것들)
target_patterns = [name for name, hits in results.items() if len(hits) > 0]

for name in target_patterns:
    hits = results[name]
    print(f"\n{'═'*60}")
    print(f"  [{name}]  총 {len(hits)}개")
    print(f"{'─'*60}")
    for page_num, line in hits[:8]:  # 최대 8개만 출력
        print(f"  p.{page_num:3d} | {line[:70]}")
    if len(hits) > 8:
        print(f"  ... 외 {len(hits)-8}개")

### 4. 2단 레이아웃 오탐 필터 설계

In [ ]:
# 셀 4: 오탐 필터 함수 정의
# Step 2에서 확인한 2단 합침 문제와 중복 텍스트를 걸러냅니다

def is_valid_heading(line: str) -> bool:
    """
    True  → 실제 헤딩으로 처리
    False → 오탐으로 제거
    """
    line = line.strip()

    # 1. 너무 짧음 (페이지 번호, 단독 숫자)
    if len(line) <= 3:
        return False

    # 2. 순수 숫자 (페이지 번호)
    if re.match(r"^\d+$", line):
        return False

    # 3. 너무 긺 — 2단 합침 줄 (본문 두 줄이 붙은 경우)
    if len(line) > 120:
        return False

    # 4. {WGI ...} 참조 포함 — 본문 줄
    if re.search(r"\{W[GI]+\s+[A-Z]+", line):
        return False

    # 5. 그림/표 캡션 — 헤딩 아님
    if re.match(r"^(그림|표)\s*[\d.]+", line):
        return False

    # 6. 영문+숫자 저자 목록 줄 (판권 페이지)
    if re.match(r"^[A-Z]{2,},\s+[A-Z][a-z]", line):
        return False

    return True

# 필터 적용 후 유효 헤딩 추출
valid_headings = []
for page_num, line in lines_with_page:
    for name, pat in HEADING_PATTERNS.items():
        if name in ("그림", "표"):      # 오탐 확인용 패턴은 제외
            continue
        if re.match(pat, line) and is_valid_heading(line):
            valid_headings.append({
                "page": page_num,
                "pattern": name,
                "text": line
            })
            break

print(f"필터 전 전체 매칭: {sum(len(v) for v in results.values())}개")
print(f"필터 후 유효 헤딩: {len(valid_headings)}개")
print()
print(f"{'패턴명':18s} {'수':>5s}")
print("─" * 30)
from collections import Counter
cnt = Counter(h["pattern"] for h in valid_headings)
for name, c in cnt.most_common():
    print(f"  {name:16s} {c:4d}개")

### 5. 헤딩 순서 및 계층 구조 육안 확인
- 확인할 것: 페이지 순서대로 챕터 → 절 → 항이 자연스럽게 이어지는지, 중간에 이상한 헤딩이 끼어있지 않은지.

In [ ]:
# 셀 5: 헤딩 순서대로 출력 — 계층 구조가 자연스러운지 확인
print(f"{'페이지':>5s}  {'패턴':16s}  헤딩 텍스트")
print("─" * 80)
for h in valid_headings:
    # 계층 깊이 표시 (들여쓰기)
    indent = {
        "섹션번호": "",   "SPM섹션": "",  "부속서": "",
        "숫자점": "  ",
        "숫자점숫자": "    ",
        "숫자점숫자점숫자": "      ",
        "Box": "  ",     "CrossSection": "  ",
        "핵심발견": "  ", "자주묻는질문": "  ",
        "알파벳점": "  ", "참고문헌": "",  "용어해설": "",
    }.get(h["pattern"], "")
    print(f"  p.{h['page']:3d}  {h['pattern']:16s}  {indent}{h['text'][:60]}")

### 6. 최종 패턴 확정 및 저장


In [ ]:
# 셀 6: 최종 패턴 확정 및 JSON 저장
# ── 셀 5 확인 후 오탐 패턴을 EXCLUDE_PATTERNS에 추가하세요 ──
EXCLUDE_PATTERNS = set()  # 예: {"알파벳점"} — 오탐이 많으면 추가

final_headings = [h for h in valid_headings if h["pattern"] not in EXCLUDE_PATTERNS]

# 저장
headings_path = f"{OUTPUT_DIR}/ipcc_headings.json"
with open(headings_path, "w", encoding="utf-8") as f:
    json.dump(final_headings, f, ensure_ascii=False, indent=2)

print(f"✓ 헤딩 목록 저장: {headings_path}")
print(f"  최종 헤딩 수: {len(final_headings)}개")
print()

# 메모리 문서용 요약
print(f"""
╔══════════════════════════════════════════════════╗
  Day 1 — Step 3 구조 패턴 분석 결과 요약
╚══════════════════════════════════════════════════╝

확정된 헤딩 패턴:
{chr(10).join(f"  ✓ {name}: {cnt[name]}개" for name in cnt if name not in EXCLUDE_PATTERNS)}

제외된 패턴: {EXCLUDE_PATTERNS if EXCLUDE_PATTERNS else "없음"}

주요 발견:
  - 2단 레이아웃 합침 줄 → 길이 120자 초과 필터로 처리
  - 판권 페이지 저자 목록 → 영문 대문자 콤마 패턴 필터로 처리
  - 그림/표 캡션 → 별도 오탐 패턴으로 분리 처리

저장 파일:
  ✓ ipcc_headings.json  (Day 2 청킹 로직 입력값)

다음 단계: Day 2 — 구조 기반 청킹 로직 구현
""")

### 7. 오탐 필터 추가 및 재처리

In [ ]:
# 셀 7: SPM섹션·Box 오탐 필터 추가 후 헤딩 재확정

# 기존 is_valid_heading에 추가할 규칙 2개
REFERENCE_KEYWORDS = ("WGII", "WGIII", "WGI ", "SR1.5", "SRCCL", "SROCC")

def is_valid_heading_v2(line: str, pattern_name: str) -> bool:
    line = line.strip()

    # 기존 필터 유지
    if len(line) <= 3:                          return False
    if re.match(r"^\d+$", line):                return False
    if len(line) > 120:                         return False
    if re.search(r"\{W[GI]+\s+[A-Z]+", line):  return False
    if re.match(r"^(그림|표)\s*[\d.]+", line):  return False
    if re.match(r"^[A-Z]{2,},\s+[A-Z][a-z]", line): return False

    # ── 신규 필터 1: SPM섹션 오탐 ──────────────────────
    # "SPM X.X, WGII ..." 형태 — 참조 줄이지 헤딩이 아님
    if pattern_name == "SPM섹션":
        # 실제 헤딩: "SPM.A 현황 및 추세" 처럼 한글 제목이 이어짐
        # 오탐:     "SPM D.1.1, WGII ..." 처럼 참조 키워드가 이어짐
        if any(kw in line for kw in REFERENCE_KEYWORDS):
            return False
        if re.match(r"^SPM\s+[A-Z]\.\d", line):  # SPM D.1.1 형태
            return False

    # ── 신규 필터 2: Box 오탐 ───────────────────────────
    # 실제 Box 헤딩: "Box 1.1 제목텍스트" — 숫자.숫자 뒤 공백+텍스트
    # 오탐:         "Box.2, 그림 ..."    — 점 바로 뒤 쉼표
    if pattern_name == "Box":
        if re.match(r"^Box\.\d+,", line):         # Box.N, 형태
            return False
        if "그림" in line and len(line) > 20:      # 그림 참조 포함 긴 줄
            return False

    return True

# 전체 재처리
valid_headings_v2 = []
for page_num, line in lines_with_page:
    for name, pat in HEADING_PATTERNS.items():
        if name in ("그림", "표"):
            continue
        if re.match(pat, line) and is_valid_heading_v2(line, name):
            valid_headings_v2.append({
                "page": page_num,
                "pattern": name,
                "text": line
            })
            break

# 변화 확인
from collections import Counter
cnt_v2 = Counter(h["pattern"] for h in valid_headings_v2)

print(f"{'패턴명':18s} {'v1':>5s} → {'v2':>5s}  {'변화':>6s}")
print("─" * 45)
all_names = set(list(cnt.keys()) + list(cnt_v2.keys()))
for name in sorted(all_names):
    before = cnt.get(name, 0)
    after  = cnt_v2.get(name, 0)
    diff   = after - before
    flag   = f"  ▼ -{abs(diff)}" if diff < 0 else ("  ─" if diff == 0 else f"  ▲ +{diff}")
    print(f"  {name:16s} {before:5d} → {after:5d}{flag}")

print()
print(f"  전체: {len(valid_headings)} → {len(valid_headings_v2)}개  (오탐 {len(valid_headings)-len(valid_headings_v2)}개 제거)")

### 8. 최종 저장 및 Day 1 완료 선언

In [ ]:
# 셀 8: 최종 헤딩 저장 및 Day 1 완료 요약

# ipcc_headings.json 덮어쓰기
with open(headings_path, "w", encoding="utf-8") as f:
    json.dump(valid_headings_v2, f, ensure_ascii=False, indent=2)

print(f"✓ ipcc_headings.json 업데이트 완료")
print(f"  최종 헤딩 수: {len(valid_headings_v2)}개")
print()

# 최종 샘플 — 핵심 헤딩 패턴만 추려서 확인
print("── 최종 헤딩 샘플 (실제 제목 헤딩만) ─────────────────")
title_patterns = {"섹션번호", "숫자점숫자", "숫자점숫자점숫자", "숫자점", "Box", "CrossSection", "부속서"}
for h in valid_headings_v2:
    if h["pattern"] in title_patterns:
        print(f"  p.{h['page']:3d}  {h['pattern']:18s}  {h['text'][:55]}")

print(f"""
╔══════════════════════════════════════════════════╗
  Day 1 완료 요약
╚══════════════════════════════════════════════════╝

Step 1  PDF 진단          ✓ 완료
Step 2  텍스트 추출        ✓ 완료  (523,738자 / 169페이지)
Step 3  구조 패턴 분석     ✓ 완료  (최종 {len(valid_headings_v2)}개 헤딩)

저장 파일:
  ipcc_raw.txt        전체 텍스트 (페이지 구분자 포함)
  ipcc_pages.json     페이지별 데이터 (Day 2 재사용)
  ipcc_headings.json  확정 헤딩 목록 (Day 2 청킹 입력값)

Day 2에서 할 것:
  ipcc_headings.json를 기준으로 헤딩 사이 텍스트를
  청크 단위로 자르는 구조 기반 청킹 로직 구현
""")

Important update
On April 24 we'll start using GitHub Copilot interaction data for AI model training unless you opt out. Review this update and manage your preferences in your GitHub account settings.

https://www.threads.com/@jay_ai__/post/DMZFfFrPMu0/%EA%B5%AC%EA%B8%80%EB%A9%94%ED%83%80-%EC%97%94%EC%A7%80%EB%8B%88%EC%96%B4%EB%93%A4%EB%8F%84-%EC%82%AC%EC%9A%A9%ED%95%98%EB%8A%94-rag-%EC%B2%AD%ED%82%B9-%EC%A0%84%EB%9E%B5-%EC%B4%9D%EC%A0%95%EB%A6%AC-%EB%AC%B4%EC%A1%B0%EA%B1%B4%EC%A0%81%EC%9D%B8-%EC%B2%AD%ED%82%B9%EC%9D%80-%EB%A7%90%EB%A6%AC%EB%8A%94-%ED%8E%B8%EC%9D%B4%EC%A7%80%EB%A7%8C%EC%9E%98-%EC%84%A0%ED%83%9D%ED%95%9C-%EC%A0%84%EB%9E%B5%EC%A0%81%EC%9D%B8-%EC%B2%AD%ED%82%B9%EC%9D%80-rag%EC%9D%98-%ED%98%B8%ED%9D%A1-%EB%A7%A5%EA%B3%BC%EB%8F%84-%EA%B0%99%EC%8A%B5%EB%8B%88%EB%8B%A4